## Downloads Documents

In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import os

# Base URL of the website
base_url = 'https://rbi.org.in/Scripts/AnnualPublications.aspx?head=State+Finances+%3a+A+Study+of+Budgets'

# Set headers to mimic a browser request
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Define the path where you want to store the files
download_folder = r'C:/Users/chandresh.kanyal/Downloads'  # Change this to your desired folder path

# Ensure the directory exists
if not os.path.exists(download_folder):
    os.makedirs(download_folder)

# Send a GET request to the website
response = requests.get(base_url, headers=headers)

# Check if the request was successful
if response.status_code == 200:
    # Parse the HTML content using BeautifulSoup
    soup = BeautifulSoup(response.content, 'html.parser')

    # Find the table that contains the row for Statement 21
    table = soup.find('table')  # You may need to adjust this if there are multiple tables

    # Iterate through each row in the table to find the relevant one
    for row in table.find_all('tr'):
        cells = row.find_all('td')
        if len(cells) > 0 and 'Statement 34: Revenue Expenditure of State Governments and UTs' in cells[0].text:
            # This is the correct row
            print("Found the row for Statement 21")

            # Look for the download links (Excel and PDF)
            excel_link = None
            pdf_link = None

            # Iterate through the cells to find the download links (Excel and PDF)
            for cell in cells:
                # Find all anchor tags with the 'href' attribute (links)
                links = cell.find_all('a', href=True)
                for link in links:
                    href = link['href']
                    print(f"Found link: {href}")  # Debugging: print each link found
                    
                    # Check if the link is an Excel or PDF file
                    if '.xlsx' in href:
                        excel_link = urljoin(base_url, href)  # Resolve relative URL
                    elif '.pdf' in href:
                        pdf_link = urljoin(base_url, href)  # Resolve relative URL

            # Check and download the Excel file if the link is found
            if excel_link:
                print(f"Found Excel file link: {excel_link}")
                try:
                    excel_response = requests.get(excel_link, headers=headers, allow_redirects=True)
                    print(f"Excel response status: {excel_response.status_code}")  # Debugging response status
                    excel_response.raise_for_status()  # Raise an error for bad status codes
                    excel_file_path = os.path.join(download_folder, 'Statement_21_Market_Borrowings.xlsx')
                    with open(excel_file_path, 'wb') as f:
                        f.write(excel_response.content)
                    print(f"Excel file downloaded successfully to: {excel_file_path}")
                except requests.exceptions.RequestException as e:
                    print(f"Error downloading Excel file: {e}")

            # Check and download the PDF file if the link is found
            if pdf_link:
                print(f"Found PDF file link: {pdf_link}")
                try:
                    pdf_response = requests.get(pdf_link, headers=headers, allow_redirects=True)
                    print(f"PDF response status: {pdf_response.status_code}")  # Debugging response status
                    pdf_response.raise_for_status()  # Raise an error for bad status codes
                    pdf_file_path = os.path.join(download_folder, 'Statement_21_Market_Borrowings.pdf')
                    with open(pdf_file_path, 'wb') as f:
                        f.write(pdf_response.content)
                    print(f"PDF file downloaded successfully to: {pdf_file_path}")
                except requests.exceptions.RequestException as e:
                    print(f"Error downloading PDF file: {e}")
            
            # Break the loop after downloading
            break
else:
    print(f"Failed to retrieve the webpage. Status code: {response.status_code}")


Found the row for Statement 21
Found link: PublicationsView.aspx?id=22299
Found link: https://rbidocs.rbi.org.in/rdocs/Publications/DOCs/34_STATEMENT111220233311022A46834E8CAECF8F88059DC8BF.XLSX
Found link: https://rbidocs.rbi.org.in/rdocs/Publications/PDFs/34_STATEMENT111220233311022A46834E8CAECF8F88059DC8BF.PDF


## Converting PDF to Desired Format Excel

In [2]:
import pdfplumber
import pandas as pd

# Path to the PDF file
pdf_file_path = r"C:/Users/chandresh.kanyal/Downloads/34_STATEMENT111220233311022A46834E8CAECF8F88059DC8BF.PDF"

# Output Excel file path
output_excel_path = r"C:/Users/chandresh.kanyal/Desktop/excel_converted34.xlsx"

# Parse the PDF for tables
try:
    with pdfplumber.open(pdf_file_path) as pdf:
        tables = []
        for page_num, page in enumerate(pdf.pages, start=1):
            # Extract table from the current page
            table = page.extract_table()
            if table:
                # Convert the table to a DataFrame
                df = pd.DataFrame(table[1:], columns=table[0])  # Use first row as headers
                tables.append((f"Page_{page_num}", df))

        if tables:
            # Save all tables to an Excel file
            with pd.ExcelWriter(output_excel_path) as writer:
                for sheet_name, df in tables:
                    df.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"All tables exported successfully to: {output_excel_path}")
        else:
            print("No tables found in the PDF.")
except Exception as e:
    print(f"Error parsing the PDF: {e}")


All tables exported successfully to: C:/Users/chandresh.kanyal/Desktop/excel_converted34.xlsx


In [3]:
# Path to the Excel file
file_path = "C:/Users/chandresh.kanyal/Desktop/excel_converted34.xlsx"  # Replace with the actual path to your Excel file

# Load the Excel file into a DataFrame
df = pd.read_excel(file_path)

# Print the DataFrame
print(df)


                                            State/UT  \
0                                                NaN   
1                                                  1   
2  1. Andhra Pradesh\n2. Arunachal Pradesh\n3. As...   
3                                All States and UTs#   

                                             2021-22  \
0                                          RE/\nGSDP   
1                                                  2   
2  14.0\n45.1\n20.0\n24.5\n18.5\n17.2\n8.3\n11.3\...   
3                                               14.2   

                                          Unnamed: 2  \
0                                         DRE/\nGSDP   
1                                                  3   
2  9.1\n31.4\n11.1\n17.0\n12.9\n11.2\n5.3\n6.9\n1...   
3                                                8.7   

                                          Unnamed: 3  \
0                                        NDRE/\nGSDP   
1                                            

In [6]:
import pandas as pd

# Load the new Excel file into a DataFrame
input_file_path = "C:/Users/chandresh.kanyal/Desktop/excel_converted34.xlsx"  # Replace with the path to your new file
df = pd.read_excel(input_file_path)

# Step 1: Clean the DataFrame
# Remove the first two rows (assuming they are metadata)
df_cleaned = df.iloc[2:].reset_index(drop=True)

# Extract state names from the "State/UT" column
state_column = df_cleaned['State/UT'].iloc[0]
states = state_column.split('\n')  # Split by newlines to extract individual state names

# Drop the "State/UT" column (will add it back later)
df_cleaned = df_cleaned.drop(columns=['State/UT'])

# Step 2: Define periods and their corresponding columns
periods_and_columns = [
    ('2021-22', '2021-22', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'),
    ('2022-23 (BE)', '2022-23 (BE)', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9'),
    ('2022-23 (RE)', '2022-23 (RE)', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14'),
    ('2023-24 (BE)', '2023-24 (BE)', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19'),
]

# Step 3: Transform data
final_rows = []
for period, _, gross_col, repayment_col, ip_col in periods_and_columns:
    # Extract data for "Gross Amount Raised", "Repayments" and "IP/GSDP"
    gross_values = df_cleaned[gross_col].iloc[0].split('\n')
    repayment_values = df_cleaned[repayment_col].iloc[0].split('\n')
    ip_values = df_cleaned[ip_col].iloc[0].split('\n')
    
    # Combine state, period, gross amount, repayment, and IP/GSDP into rows
    for state, gross, repayment, ip in zip(states, gross_values, repayment_values, ip_values):
        final_rows.append([state.strip(), period, gross.strip(), repayment.strip(), ip.strip()])

# Step 4: Create the transformed DataFrame
transformed_df = pd.DataFrame(final_rows, columns=['State/UT', 'Period', 'Gross Amount Raised', 'Repayments', 'IP/GSDP'])
# Step 5: Export to Excel (optional)
output_file_path = "C:/Users/chandresh.kanyal/Desktop/statement-34_transformed.xlsx"  # Replace with desired output path
transformed_df.to_excel(output_file_path, index=False)
print(f"Transformed data saved to {output_file_path}")
transformed_df


Transformed data saved to C:/Users/chandresh.kanyal/Desktop/statement-34_transformed.xlsx


,State/UT,Period,Gross Amount Raised,Repayments,IP/GSDP
0,1. Andhra Pradesh,2021-22,9.1,4.9,2.0
1,2. Arunachal Pradesh,2021-22,31.4,13.8,2.2
2,3. Assam,2021-22,11.1,8.8,1.5
3,4. Bihar,2021-22,17.0,7.5,2.1
4,5. Chhattisgarh,2021-22,12.9,5.3,1.5
...,...,...,...,...,...
119,27. Uttarakhand,2023-24 (BE),9.0,5.9,1.9
120,28. West Bengal,2023-24 (BE),9.2,4.9,2.5
121,29. Jammu and Kashmir,2023-24 (BE),15.5,14.7,3.8
122,30. NCT Delhi,2023-24 (BE),3.8,0.8,0.3
